In [1]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import time
import logging

C:\Users\schak\AppData\Roaming\Python\Python39\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\schak\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
DB_URL = (
    "mssql+pyodbc://@localhost\\SQLEXPRESS/StockAnalystTrack"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

MC_URL = "https://priceapi.moneycontrol.com/pricefeed/nse/equitycash/{}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

SLEEP = 0.5   # rate limit (seconds)

In [3]:
logging.basicConfig(
    filename="ticker_fill.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

In [4]:
engine = create_engine(DB_URL)

In [5]:
def get_mc_data(mc_code):
    url = MC_URL.format(mc_code)
    r = requests.get(url,headers = headers, timeout=10)
    r.raise_for_status()
    return r.json()


def extract_nseid(js):
    return js.get("data", {}).get("NSEID")

In [6]:
with engine.connect() as conn:

    df = pd.read_sql("""
        SELECT *
        FROM nseticker
        WHERE nse_ticker_yfinance = 'NULL'
    """, conn)

print(f"Found {len(df)} rows to update")

Found 0 rows to update


In [13]:
updated = 0
failed = 0


with engine.begin() as conn:
    for _, row in df.iterrows():

        #rid = row["id"]
        mc = row["mc_ticker"]

        try:
            js = get_mc_data(mc)
            nseid = extract_nsei3d(js)

            if not nseid:
                raise ValueError("No NSEID found")

            yahoo = f"{nseid}.NS"

            conn.execute(
                text("""
                    UPDATE nseticker
                    SET nse_ticker_yfinance = :ticker
                    WHERE mc_ticker = :id
                """),
                {"ticker": yahoo, "id": mc}
            )

            updated += 1
            logging.info(f"Updated {mc} -> {yahoo}")
            print(f"Inserted record for {mc} -> {yahoo}")

        except Exception as e:

            failed += 1
            logging.error(f"Failed {mc}: {e}")

        time.sleep(SLEEP)


print(f"Updated: {updated}")
print(f"Failed : {failed}")

Inserted record for AF32 -> AAVAS.NS
Inserted record for ACC -> ACC.NS
Inserted record for AIL10 -> CPPLUS.NS
Inserted record for AEL09 -> AJAXENGG.NS
Inserted record for ADP03 -> AKUMS.NS
Inserted record for AP29 -> APLLTD.NS
Inserted record for AL16 -> ALKEM.NS
Inserted record for AAC -> ALKYLAMINE.NS
Inserted record for APT02 -> ASTRAL.NS
Inserted record for ASF03 -> AUBANK.NS
Inserted record for UTI10 -> AXISBANK.NS
Inserted record for AEL03 -> AZAD.NS
Inserted record for BWI02 -> BANSALWIRE.NS
Inserted record for BE03 -> BEL.NS
Inserted record for BHE -> BHEL.NS
Inserted record for BPC -> BPCL.NS
Inserted record for BFI01 -> BIKAJI.NS
Inserted record for BGV -> GROWW.NS
Inserted record for BJH -> BLUEJET.NS
Inserted record for CHL02 -> CANHLIFE.NS
Inserted record for CI01 -> CASTROLIND.NS
Inserted record for CWL -> CELLO.NS
Inserted record for MOF -> CERA.NS
Inserted record for C -> CIPLA.NS
Inserted record for CI02 -> CUMMINSIND.NS
Inserted record for DI -> DABUR.NS
Inserted reco